In [ ]:
!pip install surprise

In [ ]:
import pandas as pd
import numpy as np
import pickle
from surprise import Dataset, Reader, SVD

# 전처리된 마스터 데이터셋 로드
file_path = 'hybrid_drop1.csv'
try:
    df = pd.read_csv(file_path)
    print(f"'{file_path}' 로드 완료. (행: {len(df)})")
except FileNotFoundError:
    print(f"오류: '{file_path}' 파일을 찾을 수 없습니다. 이전 전처리 셀을 실행하세요.")
    raise

print("\n CF 모델 학습 시작 (Spotify ID 기반)...")
reader = Reader(rating_scale=(0, 1))
data = Dataset.load_from_df(df[['user_id', 'spotify_id', 'rating']], reader)

trainset = data.build_full_trainset()
model_cf = SVD(n_factors=100, n_epochs=20, random_state=42, verbose=False)
model_cf.fit(trainset)

with open('cf_model_final.pkl', 'wb') as f:
    pickle.dump(model_cf, f)
print("--- 'cf_model_final.pkl' 저장 완료 ---")


print("\n트랙 메타 DB 구축 시작 (Spotify ID 기반)...")

# 트랙별 인기도(평가 횟수) 계산
rating_counts = df['spotify_id'].value_counts().reset_index()
rating_counts.columns = ['spotify_id', 'total_rating_count']

# 트랙별 V/A 정보 추출 (중복 제거)
track_info = df[['spotify_id', 'valence', 'energy']].drop_duplicates(subset=['spotify_id'])

# 인기도와 V/A를 병합
meta_df = pd.merge(track_info, rating_counts, on='spotify_id', how='left')
meta_df['total_rating_count'] = meta_df['total_rating_count'].fillna(0).astype(int)

# 딕셔너리 형태로 변환 (키가 Spotify ID)
track_meta_db = meta_df.set_index('spotify_id').to_dict('index')

with open('track_meta_db.pkl', 'wb') as f:
    pickle.dump(track_meta_db, f)

print(f"--- 'track_meta_db.pkl' 저장 완료 (총 {len(track_meta_db)}개 트랙) ---")
print("\n--- 2개 자산 생성 모두 완료 ---")

In [ ]:
import pandas as pd
import numpy as np
import pickle
import lightgbm as lgb
from sklearn.model_selection import train_test_split as sklearn_split
from surprise import SVD

final_cf_model = pickle.load(open('cf_model_final.pkl', 'rb'))
track_meta_db = pickle.load(open('track_meta_db.pkl', 'rb'))
print("   - cf_model_final.pkl, track_meta_db.pkl 로드 완료")

# 학습 데이터셋 로드
print("학습 데이터셋 로딩 중 ---")
try:
    df_train = pd.read_csv('hybrid_drop1.csv')
    print(f"   - 학습 데이터(hybrid_drop1) 로드 완료: {len(df_train)}")
except FileNotFoundError:
    print("오류: 'hybrid_drop1.csv' 파일을 찾을 수 없습니다.")
    raise

print("\n-LTR 피쳐 엔지니어링 시작 ")
USER_EMOTION_V = 0.5 # 학습용 가상 감정
USER_EMOTION_A = 0.5

def build_ltr_features(dataframe, df_name, cf_model, meta_db, user_emotion_v, user_emotion_a):
    print(f"--- {df_name} 피쳐 구축 시작 ---")

    # taste_score
    print("   - 피쳐 1: 'taste_score' 계산 중...")
    dataframe['taste_score'] = dataframe.apply(
        lambda row: cf_model.predict(row['user_id'], row['spotify_id']).est,
        axis=1
    )

    # emotion_score, novelty_score
    print("   - 피쳐 2, 3: 'emotion_score', 'novelty_score' 계산 중...")

    def calculate_cb_scores(row, meta_db_ref, user_emov, user_emoa):
        track_id = row['spotify_id']
        meta = meta_db_ref.get(track_id)

        track_valence = row['valence']
        track_energy = row['energy']

        distance = np.sqrt((user_emov - track_valence)**2 + (user_emoa - track_energy)**2)
        emotion_score = 1 / (1 + distance)

        novelty_score = np.nan
        if meta and 'total_rating_count' in meta:
            novelty_score = 1 / (meta['total_rating_count'] + 1)

        return emotion_score, novelty_score, (row['rating'] >= 0.5)

    features_df = dataframe.apply(
        lambda row: calculate_cb_scores(row, meta_db, user_emotion_v, user_emotion_a),
        axis=1, result_type='expand'
    )
    dataframe['emotion_score'] = features_df[0]
    dataframe['novelty_score'] = features_df[1]

    # Label 및 Query 정리
    print("   - 피쳐 4, 5: 'Label', 'Query' 정리 중...")
    dataframe['label'] = features_df[2].astype(int)
    dataframe['query_id'] = dataframe['user_id'].astype('category').cat.codes

    # 최종 LTR 데이터셋 완성 (결측치 제거)
    ltr_dataframe = dataframe.dropna(subset=['taste_score', 'emotion_score', 'novelty_score', 'label'])
    print(f"{df_name} LTR 피쳐 구축 완료: {len(ltr_dataframe)}개")
    return ltr_dataframe

# 학습 데이터에 대해서만 피쳐 구축 실행
ltr_train_df = build_ltr_features(df_train.copy(), "Train_DF", final_cf_model, track_meta_db, USER_EMOTION_V, USER_EMOTION_A)
del df_train # 메모리 확보

print("\n[자산 3] 최종 모델 (API 서빙용) 학습 시작 (평가 생략) ---")

feature_columns = ['taste_score', 'emotion_score', 'novelty_score']

X_full = ltr_train_df[feature_columns]
y_full = ltr_train_df['label']
q_full = ltr_train_df.groupby('query_id').size().values

# 평가를 안 했으므로, 'n_estimators' 값을 500 등으로 고정
FIXED_N_ESTIMATORS = 500

final_ranking_model = lgb.LGBMRanker(
    objective='lambdarank',
    device='cpu',
    random_state=42,
    n_estimators=FIXED_N_ESTIMATORS, # 고정된 값 사용
    learning_rate=0.05,
    verbose = -1
)

# 전체 학습 데이터로 'fit'만 실행
print(f"   - {FIXED_N_ESTIMATORS}개 트리로 최종 모델 학습 중...")
final_ranking_model.fit(
    X=X_full,
    y=y_full,
    group=q_full
)

# [자산 3] 저장
with open('ranking_model_final.pkl', 'wb') as f:
    pickle.dump(final_ranking_model, f)
print("--- 'ranking_model_final.pkl' 저장 완료 ---")


In [ ]:
from google.colab import drive
import os

# 1. Google 드라이브 연결 (마운트)
drive.mount('/content/drive')

# 2. 복사할 파일 목록
file_names = ['cf_model_final.pkl',
              'track_meta_db.pkl',
              'ranking_model_final.pkl']

# 3. 내 드라이브(MyDrive) 최상위 폴더로 복사
print("\n--- 3개 파일 복사 시작 ---")
for file in file_names:
    source_path = f"/content/{file}"  # 코랩 현재 위치
    target_path = f"/content/drive/MyDrive/{file}" # 내 드라이브 위치

    if os.path.exists(source_path):
        !cp "{source_path}" "{target_path}"
        print(f"✅ '{file}' 복사 완료.")
    else:
        print(f"❌ 오류: '{file}'을(를) 찾을 수 없습니다. [STEP 1]과 [STEP 2]가 정상 실행되었는지 확인하세요.")

print("\n--- 구글 드라이브로 복사 완료 ---")